In [ ]:
#Import Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load Dataset

df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

print("Dataset Loaded Successfully")
print(df.head())


In [ ]:
#Dataset Information
print("\nShape of Dataset:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nDataset Info:")
print(df.info())

In [ ]:
#Check Null values
print(df.isnull().sum())

In [ ]:
#Check Duplicate Values

print(df.duplicated().sum())

In [ ]:
#Dataset Statistical summary
df.describe()

In [ ]:
#Exploratory Data Analysis (EDA) :

#Deliveries Trend

plt.figure(figsize=(10,5))

sns.lineplot(
    x='Year',
    y='Estimated_Deliveries',
    data=df
)

plt.title("Tesla Deliveries Over Years")

plt.show()

#Deliveries Trend Graph Observation:

Tesla estimated deliveries show a generally increasing tred over the years. But a slight drop can be observed during the year 2020, and the deliveries recovered again in the following years.

In [ ]:
#Region Wise Deliveries

plt.figure(figsize=(12,5))

sns.barplot(
    x='Region',
    y='Estimated_Deliveries',
    data=df 
)

plt.xticks(rotation=45)
plt.title("Region Wise Deliveires")
plt.show()

#Region Wise Deliveries Graph Observation:
The estimated deliveries are relatively balanced across different regions. Middle East and Asia regions show slightly higher deliveries compared to other regions.

In [ ]:
#Correlation Heatmap

plt.figure(figsize=(10,7))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()

#Correlation Heatmap Observation:

Estimated deliveries and production units have a very high positive correlation.

In [ ]:
#Distribution Plot

plt.figure(figsize=(10,5))

sns.histplot(
    df['Estimated_Deliveries'],
    bins=30,
    kde=True
)

plt.title("Distribution of Estimated Deliveries")

plt.show()

#Distribution Plot Observation:
The distributiion of estimated deliveries appears normal with slight right skewness. Most delivery values are focused between 7000  and 15000.

In [ ]:
#Boxplot

plt.figure(figsize=(10,5))

sns.boxplot(
    x=df['Estimated_Deliveries']
)

plt.title("Outlier Detection")

plt.show()

#Boxplot Observation:

Somme Outliers are present in the estimated deliveries column. Most of the data points are distributed within a range.

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    df.groupby('Year')['Estimated_Deliveries'].mean(),
    marker='o',
    label='Deliveries'
)

plt.plot(
    df.groupby('Year')['Production_Units'].mean(),
    marker='o',
    label='Production'
)

plt.legend()

plt.title("Production vs Deliveries Trend")

plt.show()

# Production vs Deliveries Trend 
# Observation:
Production and delivery trends move closellly together over  time, indicating strong operational alignent.

In [ ]:
#Feature Engineering

encoder = LabelEncoder()

df['Region'] = encoder.fit_transform(df['Region'])
df['Model'] = encoder.fit_transform(df['Model'])
df['Source_Type'] = encoder.fit_transform(df['Source_Type'])

print("Categorical Columns Encoded")

In [ ]:
#Features and Target Variable

x = df.drop('Estimated_Deliveries', axis=1)

y = df['Estimated_Deliveries']

In [ ]:
#Train_Test_Split the data

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
#Linear Regression Model

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

print("linear Regression Model Trained")

In [ ]:
#Random Forest Regressor

rf_model = RandomForestRegressor(
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Model Trained")

In [ ]:
#Model Evauation

print("--------Linear Regression Evaluation--------")

print("MAE:", mean_absolute_error(y_test, lr_pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))

print("R2 Score:", r2_score(y_test, lr_pred))

print("\n--------Random Forest Evaluation--------")

print("MAE:",
      mean_absolute_error(y_test, rf_pred))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, rf_pred)))

print("R2 Score:",
      r2_score(y_test, rf_pred))



In [ ]:
#HyperParameter  Tuning

params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=params,
    cv=3
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

In [ ]:
#Time series Forecasting

yearly_data = df.groupby("Year")['Estimated_Deliveries'].sum()

model = ARIMA(yearly_data, order=(1,1,1))

model_fit = model.fit()

forecast = model_fit.forecast(steps=5)

forecast_df = pd.DataFrame({
    'Year': future_years,
    'Forecasted_Deliveries': forecast.values.astype(int)
})

forecast_df

In [ ]:
#Forecast Visualization

plt.figure(figsize=(10,5))

plt.plot(
    yearly_data.index,
    yearly_data.values,
    marker='o',
    label='Original Data'
)

future_years = np.arange(
    yearly_data.index.max() +1,
    yearly_data.index.max() +6
)

plt.plot(
    future_years,
    forecast,
    marker='o',
    linestyle='dashed',
    label='Forecast'
)

plt.title("Tesla Deliveries Forecast")

plt.xlabel("Year")

plt.ylabel("Estimated Deliveries")

plt.legend()
plt.show()

In [ ]:
#Model Comparison.

models = ['Linear Regression', 'Random Forest']

scores = [
    r2_score(y_test, lr_pred),
    r2_score(y_test, rf_pred)
]

plt.figure(figsize=(7,5))

plt.bar(models, scores)

plt.ylabel("R2 Score")

plt.title("Model Comparison")

plt.show()

In [ ]:
#Actual vs Predicted Graph 

plt.figure(figsize=(8,5))

plt.scatter(y_test, rf_pred)

plt.xlabel("Actual Deliveries")
plt.xlabel("Predicted Deliveries")

plt.title("Actual vs Predicted Deliveries")

plt.show()

#Observation :

Most prediction lie close to actual values, indicating good model performance.

# Scenario Analysis:

If Tesla production increases significantly in future years, estimated deliveries are also expected to increase because both varaibles show strong positive correlation.